# Sweep 17 — Training, Attention, DDI & Dimension Audit

Four gaps that were never tested across all 660 prior runs.

## Gap A — Training Schedule  (3 modes × 5 seeds)

| Cell | Mode | New runs |
|------|------|----------|
| — | visit_level_training ON ★ | Reference (Sweep 16) |
| 5 | Fixed LR — standard Adam, no visit scrambling | **5** |
| 6 | Cosine LR — CosineAnnealing + warmup, no scrambling | **5** |
| 7 | ACLM off vs ACLM lr — seed 42 re-test at champion scale | **2** |

> *"Is --visit_level_training responsible for the performance gap? And does ACLM still hurt at ~0.57 Jaccard?"*
> Run 26 found ACLM hurt by -0.007 Jac, but that was at ~0.49 (pre-champion architecture).
> Cell 7 re-tests with seed 42 only — fast sanity check to confirm or overturn Run 26 finding.

## Gap B — Historical Attention  (2 modes × 5 seeds)

| Cell | Mode | New runs |
|------|------|----------|
| — | hist ON ★ | Reference (Sweep 16) |
| 7 | hist OFF | **5** |

> *"Historical attention was locked in early and never revisited."*

## Gap C — DDI Supervision  (5 alpha values × 1 seed = fast discovery)

| Cell | ddi_alpha | New runs |
|------|-----------|----------|
| 8 | 0.0 / 0.05 / 0.1 / 0.2 / 0.5 | **5** |

> *"DDI penalty was always off — does activating it improve safety without hurting Jaccard?"*
> If a winner emerges, expand to 5 seeds manually.

## Gap D — Hidden Dimension  (3 sizes × 3 seeds)

| Cell | hidden_dim | New runs |
|------|-----------|----------|
| 9 | 128 / 256 ★ / 512 | **9** |

> *"Was 256 the right model size, or would 512 give a meaningful boost?"*

## Total new runs: 31  (~2 Kaggle sessions, +2 ACLM sanity seeds)

All runs: full system (notes + labs + copy), champion architecture
(`imdr_infused · gcn · heidr · film · L4_jac_heavy`).
Results cell auto-loads Sweep 16 as the reference row.

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn


In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    note_paths = glob.glob("/kaggle/input/**/note_embeddings_mimic3.pkl", recursive=True)
    if note_paths:
        note_src = note_paths[0]
        note_link = "./data/processed/note_embeddings_mimic3.pkl"
        if not os.path.exists(note_link):
            os.symlink(note_src, note_link)
        print(f"Note embeddings linked: {note_src}")
    else:
        print("WARNING: note_embeddings_mimic3.pkl not found")

    # Patch 1: registry.py
    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

    # Patch 2: drug_gnn.py
    gnn_path = "/kaggle/working/src/model/graph_encoders/drug_gnn.py"
    if os.path.exists(gnn_path):
        with open(gnn_path, "r") as rf:
            content = rf.read()
        if "ehr_adj" not in content:
            old = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n    ):"
            new = "        use_lab_nodes: bool = False,\n        lab_embeddings: torch.Tensor | None = None,   # (num_labs, 768)\n        ehr_adj=None,\n        **kwargs,\n    ):"
            if old in content:
                content = content.replace(old, new)
                with open(gnn_path, "w") as wf:
                    wf.write(content)
                print("  drug_gnn.py patched")

    # Patch 3: dcma_decoder.py
    dcma_path = "/kaggle/working/src/model/decoders/dcma_decoder.py"
    if os.path.exists(dcma_path):
        with open(dcma_path, "r") as rf:
            content = rf.read()
        if "note_proj" not in content:
            content = content.replace(
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)",
                "    def __init__(self, hidden_dim: int = 256, dropout: float = 0.3,\n"
                "                 note_repr_dim: int = 768, lab_repr_dim: int = 400, **kwargs):\n"
                "        super().__init__()\n"
                "        self.attn = DrugModalityAttention(hidden_dim, dropout)\n"
                "        self.hidden_dim = hidden_dim\n"
                "        self.note_proj = nn.Linear(note_repr_dim, hidden_dim)\n"
                "        self.lab_proj  = nn.Linear(lab_repr_dim,  hidden_dim)",
            )
            content = content.replace(
                "            tokens.append(F.normalize(notes_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.note_proj(F.normalize(notes_repr, dim=-1)).unsqueeze(1))",
            )
            content = content.replace(
                "            tokens.append(F.normalize(labs_repr, dim=-1).unsqueeze(1))",
                "            tokens.append(self.lab_proj(F.normalize(labs_repr, dim=-1)).unsqueeze(1))",
            )
            with open(dcma_path, "w") as wf:
                wf.write(content)
            print("  dcma_decoder.py patched")
        else:
            print("  dcma_decoder.py already patched")

print("Setup done:", os.getcwd())


In [ ]:
import yaml, subprocess, gc, json, numpy as np
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, training_overrides=None,
                preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if training_overrides:
        for k, v in training_overrides.items():
            cfg["training"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

# ── Champion architecture (same as Sweep 16) ─────────────────────────────────
CHAMPION_MODEL = {
    "ar_max_seq_len":     20,
    "note_proj_dim":      128,
    "graph_encoder_type": "drug_gnn",
    "graph_layer_type":   "gcn",
    "hgt_layers":         2,
    "encoder_type":       "imdr_infused",
    "predictor_type":     "heidr",
    "use_historical_attention": True,   # default — varied in Gap B
}
CHAMPION_TRAINING = {
    "bce_weight":           0.3,
    "soft_jaccard_weight":  1.5,
    "margin_weight":        0.05,
}
LAB_FLAGS = "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"
CHAMPION_PREP = {"lab_dim": 400}

# ── Training schedule variants (Gap A) ───────────────────────────────────────
# VISIT_ON : --visit_level_training  (already run in Sweep 16 — reference only)
# FIXED_LR : standard Adam, no visit-level scrambling
# COSINE   : CosineAnnealingLR, warmup_epochs=10, no visit-level scrambling
#
# NOTE: ACLM (per-batch LR via --aclm_mode lr) is OFF in ALL runs — train.py
# defaults --aclm_mode to "off". The diagnostic "[Run 27] ACLM ENABLED" is a
# false positive (--no_aclm is never registered in argparse). What Gap A actually
# tests is --visit_level_training (visit-level dataset scrambling) vs not.

SCHEDULES = {
    "visit_on": {
        "flags":       "--device cuda --visit_level_training --fusion_strategy film --aggregator_type last",
        "train_ov":    {},
        "label":       "visit_level_training ON  (--visit_level_training)  ← Champion",
        "note":        "Reference: use Sweep 16 results — skip unless rerunning",
    },
    "fixed_lr": {
        "flags":       "--device cuda --fusion_strategy film --aggregator_type last",
        "train_ov":    {"use_cosine_lr": False},
        "label":       "Fixed LR (standard Adam 5e-4, no visit scrambling)",
        "note":        "NEW",
    },
    "cosine_lr": {
        "flags":       "--device cuda --fusion_strategy film --aggregator_type last",
        "train_ov":    {"use_cosine_lr": True},
        "label":       "Cosine LR (CosineAnnealing + 10ep warmup, no visit scrambling)",
        "note":        "NEW",
    },
}

# ── Historical attention variants (Gap B) ────────────────────────────────────
CHAMPION_FLAGS = "--device cuda --visit_level_training --fusion_strategy film --aggregator_type last"
HIST_VARIANTS = {
    "hist_on": {
        "model_ov": {"use_historical_attention": True},
        "label":    "Historical attention ON  ← Champion",
        "note":     "Reference: use Sweep 16 results — skip unless rerunning",
    },
    "hist_off": {
        "model_ov": {"use_historical_attention": False},
        "label":    "Historical attention OFF",
        "note":     "NEW",
    },
}

SEEDS = [42, 123, 456, 789, 1024]
reports_dir = Path("experiment_reports/sweep_17_training")
results_log = []

def run_seeds(run_key, cfg_path, extra_flags):
    # Run all 5 seeds for one configuration. Skip-aware.
    import torch
    for seed in SEEDS:
        run_name = f"{run_key}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        if list(run_dir.glob("result_*.json")):
            print(f"  [SKIP] {run_name} — already done")
            results_log.append(f"SKIP: {run_name}")
            continue
        log_path = run_dir / "training_log.txt"
        cmd = (f"python -u src/train.py --config {cfg_path} "
               f"{extra_flags} {LAB_FLAGS} "
               f"--seed {seed} --results_dir {run_dir}")
        print(f"\n  --- {run_name} ---")
        try:
            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print("Config ready.  reports_dir =", reports_dir)
print()
print("Gap A — Training schedule (NEW runs only):")
for k, v in SCHEDULES.items():
    print(f"  {v['label']:<55} [{v['note']}]")
print()
print("Gap B — Historical attention (NEW runs only):")
for k, v in HIST_VARIANTS.items():
    print(f"  {v['label']:<55} [{v['note']}]")


In [ ]:
# ── Smoke test: 1 epoch × fixed_lr + hist_off (the two truly new configs) ────
Path("smoke_s17").mkdir(exist_ok=True)
smoke_cases = [
    ("fixed_lr",  SCHEDULES["fixed_lr"]["flags"],
     {**CHAMPION_TRAINING, **SCHEDULES["fixed_lr"]["train_ov"]},
     {**CHAMPION_MODEL}),
    ("hist_off",  CHAMPION_FLAGS,
     CHAMPION_TRAINING,
     {**CHAMPION_MODEL, **HIST_VARIANTS["hist_off"]["model_ov"]}),
]
smoke_results = []
for name, flags, train_ov, model_ov in smoke_cases:
    cfg_path = f"s17_smoke_{name}.yaml"
    make_config(cfg_path, model_overrides=model_ov,
                training_overrides=train_ov,
                preprocessing_overrides=CHAMPION_PREP, smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{flags} {LAB_FLAGS} "
           f"--seed 42 --results_dir smoke_s17/{name}")
    print(f"SMOKE [{name}]: running...")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    for ln in (proc.stdout + proc.stderr).strip().split("\n")[-6:]: print(ln)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {name}")
    print(f"  >>> {status}\n")
for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed.")
print("All smoke tests passed.")


In [ ]:
# ── [Gap A — 1/2] Fixed LR: standard Adam 5e-4, no ACLM, no cosine (~5h) ────
# Answers: "Is ACLM responsible for the champion's performance?"
cfg_path = "s17_fixed_lr.yaml"
make_config(cfg_path,
            model_overrides=CHAMPION_MODEL,
            training_overrides={**CHAMPION_TRAINING, **SCHEDULES["fixed_lr"]["train_ov"]},
            preprocessing_overrides=CHAMPION_PREP)
print(f"\n{'='*65}")
print(f"  Fixed LR — 5 seeds")
print(f"{'='*65}")
run_seeds("fixed_lr", cfg_path, SCHEDULES["fixed_lr"]["flags"])


In [ ]:
# ── [Gap A — 2/2] Cosine LR: CosineAnnealing + warmup, no ACLM (~5h) ────────
# Answers: "Would a cosine schedule outperform ACLM?"
cfg_path = "s17_cosine_lr.yaml"
make_config(cfg_path,
            model_overrides=CHAMPION_MODEL,
            training_overrides={**CHAMPION_TRAINING, **SCHEDULES["cosine_lr"]["train_ov"]},
            preprocessing_overrides=CHAMPION_PREP)
print(f"\n{'='*65}")
print(f"  Cosine LR — 5 seeds")
print(f"{'='*65}")
run_seeds("cosine_lr", cfg_path, SCHEDULES["cosine_lr"]["flags"])


In [ ]:
# ── [Gap A — 3/2] ACLM single-seed sanity check (~1h) ────────────────────────
# Run 26 showed ACLM hurt (-0.007 Jac) but that was at Jaccard ~0.49, before the
# champion architecture was locked. Now at ~0.57, we re-test with seed 42 only:
#   aclm_off : --aclm_mode off  + --visit_level_training  (≈ champion, fast reference)
#   aclm_lr  : --aclm_mode lr   + --visit_level_training  (actual ACLM per-batch LR)
# If aclm_lr still loses at the champion level, the Run 26 finding holds.
# NOTE: previous Sweep 17 runs showed "[Run 27] ACLM ENABLED" for all configs —
# that was a false diagnostic. This cell uses --aclm_mode explicitly so the log
# will correctly print "ACLM DISABLED" vs "ACLM ENABLED".
ACLM_DISCOVERY_SEED = 42
ACLM_MODES = [("off", "--no_aclm"), ("lr", "--aclm_mode lr")]

print(f"\n{'='*65}")
print(f"  Gap A-3 — ACLM single-seed check  (seed={ACLM_DISCOVERY_SEED})")
print(f"{'='*65}\n")

import torch, gc
for mode_name, mode_flag in ACLM_MODES:
    run_name = f"aclm_{mode_name}_seed{ACLM_DISCOVERY_SEED}"
    run_dir  = reports_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    if list(run_dir.glob("result_*.json")):
        print(f"  [SKIP] {run_name} — already done")
        results_log.append(f"SKIP: {run_name}")
        continue
    cfg_path = f"s17_aclm_{mode_name}.yaml"
    make_config(cfg_path,
                model_overrides=CHAMPION_MODEL,
                training_overrides=CHAMPION_TRAINING,
                preprocessing_overrides=CHAMPION_PREP)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{CHAMPION_FLAGS} {LAB_FLAGS} "
           f"{mode_flag} "
           f"--seed {ACLM_DISCOVERY_SEED} --results_dir {run_dir}")
    log_path = run_dir / "training_log.txt"
    print(f"  --- aclm_mode={mode_name} ---")
    try:
        with open(log_path, "w") as lf:
            proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(line, end="")
                lf.write(line)
            proc.wait()
        status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
        results_log.append(f"{status}: {run_name}")
    except Exception as e:
        results_log.append(f"CRASH: {run_name}: {e}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# ── [Gap B — 1/1] Historical attention OFF (~5h) ─────────────────────────────
# Answers: "Does within-patient historical attention actually help?"
cfg_path = "s17_hist_off.yaml"
make_config(cfg_path,
            model_overrides={**CHAMPION_MODEL, **HIST_VARIANTS["hist_off"]["model_ov"]},
            training_overrides=CHAMPION_TRAINING,
            preprocessing_overrides=CHAMPION_PREP)
print(f"\n{'='*65}")
print(f"  Historical attention OFF — 5 seeds")
print(f"{'='*65}")
run_seeds("hist_off", cfg_path, CHAMPION_FLAGS)


In [ ]:
# ── [Gap C] DDI supervision discovery: 5 alpha values × seed 42 (~5h) ────────
# ddi_alpha=0.0 is the current champion (monitoring only, no DDI loss).
# A value > 0 enables the DDI penalty during training and may push the model
# to avoid high-risk drug combinations even at a small Jaccard cost.
# Run with seed 42 only for fast discovery; if one alpha is a clear winner,
# run 5-seed validation by simply re-running this cell with SEEDS=[42,123,456,789,1024].
DDI_DISCOVERY_SEED = 42
DDI_ALPHAS = [0.0, 0.05, 0.1, 0.2, 0.5]

print(f"\n{'='*65}")
print(f"  Gap C — DDI Supervision Discovery  (seed={DDI_DISCOVERY_SEED})")
print(f"  Alphas: {DDI_ALPHAS}")
print(f"{'='*65}\n")

import torch, gc
for alpha in DDI_ALPHAS:
    run_name = f"ddi_alpha{str(alpha).replace('.','p')}_seed{DDI_DISCOVERY_SEED}"
    run_dir  = reports_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    if list(run_dir.glob("result_*.json")):
        print(f"  [SKIP] {run_name} — already done")
        results_log.append(f"SKIP: {run_name}")
        continue
    cfg_path = f"s17_ddi_{str(alpha).replace('.','p')}.yaml"
    make_config(cfg_path,
                model_overrides=CHAMPION_MODEL,
                training_overrides=CHAMPION_TRAINING,   # ddi_alpha NOT in YAML — train.py ignores it there
                preprocessing_overrides=CHAMPION_PREP)
    # CRITICAL: ddi_alpha must be passed as a CLI flag — train.py reads it via
    # argparse (args.ddi_alpha), NOT from cfg["training"]["ddi_alpha"].
    # The YAML override is silently ignored. Pass it here on the command line.
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{CHAMPION_FLAGS} {LAB_FLAGS} "
           f"--ddi_alpha {alpha} "
           f"--seed {DDI_DISCOVERY_SEED} --results_dir {run_dir}")
    log_path = run_dir / "training_log.txt"
    print(f"  --- ddi_alpha={alpha} ---")
    try:
        with open(log_path, "w") as lf:
            proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:
                print(line, end="")
                lf.write(line)
            proc.wait()
        status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
        results_log.append(f"{status}: {run_name}")
    except Exception as e:
        results_log.append(f"CRASH: {run_name}: {e}")
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# ── [Gap D] Hidden dimension: 128 / 256 / 512 × 3 seeds (~9h) ────────────────
# 256 is the current champion. Testing 128 (smaller/faster) and 512 (larger).
# 3 seeds is sufficient to see the trend; expand to 5 if 512 is a clear winner.
# NOTE: hidden_dim changes the entire model — drug GNN, temporal encoder,
#       fusion, aggregator, decoder all scale with it.
HIDDEN_DIMS  = [128, 256, 512]
HDIM_SEEDS   = [42, 123, 456]

print(f"\n{'='*65}")
print(f"  Gap D — Hidden Dimension  (dims={HIDDEN_DIMS}, seeds={HDIM_SEEDS})")
print(f"{'='*65}\n")

import torch, gc
for hdim in HIDDEN_DIMS:
    for seed in HDIM_SEEDS:
        run_name = f"hdim{hdim}_seed{seed}"
        run_dir  = reports_dir / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        if list(run_dir.glob("result_*.json")):
            print(f"  [SKIP] {run_name} — already done")
            results_log.append(f"SKIP: {run_name}")
            continue
        cfg_path = f"s17_hdim{hdim}.yaml"
        # hidden_dim drives note_proj_dim and lab_proj_dim proportionally
        note_proj = max(32, hdim // 2)   # 64 / 128 / 256
        make_config(cfg_path,
                    model_overrides={**CHAMPION_MODEL,
                                     "hidden_dim":   hdim,
                                     "note_proj_dim": note_proj},
                    training_overrides=CHAMPION_TRAINING,
                    preprocessing_overrides=CHAMPION_PREP)
        cmd = (f"python -u src/train.py --config {cfg_path} "
               f"{CHAMPION_FLAGS} {LAB_FLAGS} "
               f"--seed {seed} --results_dir {run_dir}")
        log_path = run_dir / "training_log.txt"
        print(f"\n  --- hdim={hdim}, seed={seed} ---")
        try:
            with open(log_path, "w") as lf:
                proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True)
                for line in proc.stdout:
                    print(line, end="")
                    lf.write(line)
                proc.wait()
            status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
            results_log.append(f"{status}: {run_name}")
        except Exception as e:
            results_log.append(f"CRASH: {run_name}: {e}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
total   = len(results_log)
success = sum(1 for r in results_log if "SUCCESS" in r)
failed  = sum(1 for r in results_log if "FAILED"  in r)
crashed = sum(1 for r in results_log if "CRASH"   in r)
skipped = sum(1 for r in results_log if "SKIP"    in r)
print(f"Run log: {total} entries | {success} success | {failed} failed | {crashed} crashed | {skipped} skipped")
if failed + crashed > 0:
    print("\nFailed/crashed:")
    for r in results_log:
        if "SUCCESS" not in r and "SKIP" not in r: print(f"  {r}")


In [ ]:
import json, numpy as np
from pathlib import Path

# ── Load this sweep's results + Sweep 16 champion as reference ────────────────
# Point sweep16_dir at your Sweep 16 exported results folder after merging.
sweep16_dir = Path("experiment_reports/sweep_16_champion")

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

def load_dir(d):
    runs = {}
    if not d.exists(): return runs
    for jp in sorted(d.rglob("result_*.json")):
        with open(jp) as f: data = json.load(f)
        run_name = jp.parent.name
        idx = run_name.rfind("_seed")
        if idx == -1: continue
        runs.setdefault(run_name[:idx], []).append(data)
    return runs

s17 = load_dir(reports_dir)
s16 = load_dir(sweep16_dir)

def summarize(runs_dict, key):
    runs = runs_dict.get(key, [])
    if not runs: return None
    jac   = [get_metric(d, "jaccard")  for d in runs]
    f1    = [get_metric(d, "f1")       for d in runs]
    ddi   = [get_metric(d, "DDI Rate") for d in runs]
    prauc = [get_metric(d, "PRAUC")    for d in runs]
    return dict(jac=np.mean(jac), std=np.std(jac, ddof=1) if len(jac)>1 else 0,
                f1=np.mean(f1), ddi=np.mean(ddi), prauc=np.mean(prauc), n=len(jac))

def row(label, s, champion_jac=None, mark=""):
    if s is None:
        print(f"  {label:<50}  (not yet run)")
        return None
    delta = f" ({s['jac']-champion_jac:+.4f})" if champion_jac is not None else ""
    print(f"  {label:<50} {s['jac']:.4f} {s['std']:.4f}  {s['f1']:.4f}  {s['ddi']:.4f}  {s['prauc']:.4f}  {s['n']}{delta}{mark}")
    return s["jac"]

hdr = f"  {'Configuration':<50} {'Jac':>7} {'std':>6}  {'F1':>6}  {'DDI':>6}  {'PRAUC':>7}  n"
sep = "  " + "-"*76

# ── GAP A: Training Schedule ──────────────────────────────────────────────────
print("\n" + "="*82)
print("  GAP A — Training Schedule Ablation  (full system, champion architecture)")
print("="*82)
print(hdr); print(sep)

champ = summarize(s16, "champion_full") or summarize(s17, "visit_on")
champ_jac = champ["jac"] if champ else None
row("visit_level_training ON  ← Champion",            champ,                   None,      " ★")
row("Fixed LR (Adam 5e-4, no visit scrambling)",      summarize(s17,"fixed_lr"), champ_jac)
row("Cosine LR (CosineAnneal + warmup, no scramble)", summarize(s17,"cosine_lr"),champ_jac)
print()
print("  NOTE: ACLM (--aclm_mode lr) is OFF in all rows above — train.py defaults it off.")
print("        The gap tested is --visit_level_training (scrambled batching) vs not.")
print("  Verdict: if Fixed LR ≈ Champion, visit-level scrambling is the driver.")
print("           If Fixed LR drops >0.005, visit-level scrambling is load-bearing.")

# ── GAP A-3: ACLM single-seed check ──────────────────────────────────────────
print("\n" + "="*82)
print("  GAP A-3 — ACLM Single-Seed Check  (seed 42, champion arch + visit_level_training)")
print("  Re-tests ACLM at champion Jaccard level (~0.57) — Run 26 finding was at ~0.49.")
print("="*82)
print(hdr); print(sep)
aclm_off = summarize(s17, f"aclm_off_seed42") or summarize(s16, "champion_full") or champ
row("ACLM off  (--no_aclm, standard LR)   ← expected champion",
    aclm_off, None, " ★")
row("ACLM on   (--aclm_mode lr)",
    summarize(s17, f"aclm_lr_seed42"), aclm_off["jac"] if aclm_off else None)
print()
print("  Verdict: if ACLM lr ≈ or > off: ACLM may be worth enabling at champion scale.")
print("           If ACLM lr < off by >0.003: Run 26 finding confirmed — keep ACLM off.")

# ── GAP B: Historical Attention ───────────────────────────────────────────────
print("\n" + "="*82)
print("  GAP B — Historical Attention Ablation  (full system, champion architecture)")
print("="*82)
print(hdr); print(sep)

row("Historical attention ON  ← Champion",  champ,                    None,      " ★")
row("Historical attention OFF",              summarize(s17,"hist_off"), champ_jac)
print()
print("  Verdict: delta shows how much within-patient context contributes.")
print("           A drop >0.003 justifies keeping it in the final architecture.")

print("\n" + "="*82)

# ── GAP C: DDI Supervision ────────────────────────────────────────────────────
DDI_ALPHAS = [0.0, 0.05, 0.1, 0.2, 0.5]
DDI_DISCOVERY_SEED = 42

ddi_results = []
for alpha in DDI_ALPHAS:
    key = f"ddi_alpha{str(alpha).replace('.','p')}_seed{DDI_DISCOVERY_SEED}"
    runs = []
    for jp in sorted(reports_dir.glob(f"{key}/result_*.json")):
        with open(jp) as f: runs.append(json.load(f))
    ddi_results.append((alpha, runs[0] if runs else None))

print("\n" + "="*82)
print("  GAP C — DDI Supervision Discovery  (seed 42 only, champion architecture)")
print("  ddi_alpha=0.0 is current champion. Higher alpha = stronger DDI penalty.")
print("="*82)
print(f"  {'ddi_alpha':<12} {'Jaccard':>8} {'F1':>8} {'DDI Rate':>10} {'PRAUC':>8}")
print("  " + "-"*50)
best_ddi_jac = 0.0
best_ddi_alpha = 0.0
for alpha, d in ddi_results:
    if d is None:
        print(f"  alpha={alpha:<6}       (not yet run)")
        continue
    j = get_metric(d, "jaccard"); f1 = get_metric(d, "f1")
    ddi = get_metric(d, "DDI Rate"); pr = get_metric(d, "PRAUC")
    mark = " ← current" if alpha == 0.0 else ""
    mark = " ★ best" if j > best_ddi_jac and alpha > 0 else mark
    if j > best_ddi_jac and alpha > 0:
        best_ddi_jac = j; best_ddi_alpha = alpha
    print(f"  alpha={alpha:<6}       {j:>8.4f} {f1:>8.4f} {ddi:>10.4f} {pr:>8.4f}{mark}")
print()
print(f"  -> If best alpha ({best_ddi_alpha}) beats alpha=0.0 on Jaccard: enable DDI supervision.")
print(f"     If all alphas degrade Jaccard: keep ddi_alpha=0.0 (monitoring only).")
print(f"     DDI Rate should IMPROVE (decrease) as alpha increases — that's expected.")

# ── GAP D: Hidden Dimension ───────────────────────────────────────────────────
HIDDEN_DIMS = [128, 256, 512]
HDIM_SEEDS  = [42, 123, 456]

print("\n" + "="*82)
print("  GAP D — Hidden Dimension  (3 seeds × 3 dims, champion architecture)")
print("  hidden_dim=256 is current champion.")
print("="*82)
print(f"  {'hidden_dim':<12} {'Jac':>7} {'std':>6}  {'F1':>6}  {'DDI':>6}  {'PRAUC':>7}  n  params")
print("  " + "-"*68)
for hdim in HIDDEN_DIMS:
    runs = []
    for seed in HDIM_SEEDS:
        key = f"hdim{hdim}_seed{seed}"
        for jp in sorted(reports_dir.glob(f"{key}/result_*.json")):
            with open(jp) as f: runs.append(json.load(f))
    if not runs:
        print(f"  hdim={hdim:<8}         (not yet run)")
        continue
    jac   = [get_metric(d, "jaccard")  for d in runs]
    f1    = [get_metric(d, "f1")       for d in runs]
    ddi   = [get_metric(d, "DDI Rate") for d in runs]
    prauc = [get_metric(d, "PRAUC")    for d in runs]
    # Rough param count estimate: scales ~quadratically with hidden_dim
    param_scale = {128: "~1.8M", 256: "~4.5M", 512: "~14M"}
    mark = " ← champion" if hdim == 256 else ""
    std = np.std(jac, ddof=1) if len(jac) > 1 else 0
    print(f"  hdim={hdim:<8}  {np.mean(jac):.4f} {std:.4f}  {np.mean(f1):.4f}  "
          f"{np.mean(ddi):.4f}  {np.mean(prauc):.4f}  {len(jac)}  {param_scale.get(hdim,'?')}{mark}")

print("\n  -> If hdim=512 > hdim=256 by >0.003: consider upgrading (at 3x param cost).")
print("     If hdim=128 >= hdim=256: use smaller model — lower Kaggle memory and faster.")
print("="*82)


In [ ]:
import zipfile
from pathlib import Path
zip_name = "sweep_17_training_results.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(reports_dir.rglob("result_*.json")):
        zf.write(p, p.relative_to(reports_dir.parent))
    for p in sorted(reports_dir.rglob("training_log.txt")):
        zf.write(p, p.relative_to(reports_dir.parent))
print(f"Exported -> {zip_name}")
